# 543 — Package Faculty Panel for Advisor

Reads `faculty_panel_with_pools.jsonl`, re-orders columns so the most-readable fields come first, and writes a flat CSV alongside the source JSONL files.

**Source:** `tenure/tenure_pipeline/faculty_panel_with_pools.jsonl`  
**Output:** `tenure/tenure_pipeline/faculty_panel_advisor.csv`

In [ ]:
from pathlib import Path
import json
import csv

# Paths — notebook lives in tenure/, data lives in tenure/tenure_pipeline/
NB_DIR   = Path("tenure")          # adjust if running from workspace root vs tenure/
PIPE_DIR = NB_DIR / "tenure_pipeline"
SRC      = PIPE_DIR / "faculty_panel_with_pools.jsonl"
OUT_CSV  = PIPE_DIR / "faculty_panel_advisor.csv"

# Verify source exists
assert SRC.exists(), f"Source not found: {SRC}"
print(f"Source : {SRC}  ({SRC.stat().st_size / 1e6:.1f} MB)")
print(f"Output : {OUT_CSV}")

In [ ]:
# Human-readable 'keep first' columns (for advisor legibility)
KEEP_FIRST = [
    "name_display",
    "university",
    "uni_slug",
    "year",
    "rank",
    "pubs_year",
    "pubs_cumulative",
    "years_as_asst_so_far",
    "tenure_event",
    "year_of_tenure",
    "attrition",
    "censored",
    "openalex_id",
    "match_confidence",
]

# Peek at source to get ALL column names in original order
with open(SRC) as f:
    sample = json.loads(f.readline())

all_cols  = list(sample.keys())
remaining = [c for c in all_cols if c not in KEEP_FIRST]
col_order = KEEP_FIRST + remaining

print(f"Total columns : {len(col_order)}")
print("\nColumn order:")
for i, c in enumerate(col_order, 1):
    tag = "  ← priority" if c in KEEP_FIRST else ""
    print(f"  {i:2d}. {c}{tag}")

In [ ]:
rows = []
with open(SRC) as f:
    for line in f:
        r = json.loads(line)
        rows.append({k: r.get(k, "") for k in col_order})

with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=col_order)
    writer.writeheader()
    writer.writerows(rows)

print(f"Done: {len(rows):,} rows × {len(col_order)} columns")
print(f"CSV  : {OUT_CSV}  ({OUT_CSV.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import pandas as pd

df = pd.read_csv(OUT_CSV)
print(f"Shape: {df.shape}")
print(f"\nColumn list:\n{list(df.columns)}")
df.head(5)